# rpart 教程（2/3）：按 method 展开

 >  本文件在系列中的位置 ：这是 rpart 系列教程的 第 2 部分 ，承接  `01_建树与剪枝机制.ipynb`  讲透的共用机制，按  `method`  逐个展开： `class → anova → poisson → 生存` 。每个 method 只讲它 特有 的部分。
 >
 >  系列结构 ：
 > -  `00_总览与环境.ipynb` ：预备 + 附录 A（predict 速查）+ 附录 B（常见坑）
 > -  `01_建树与剪枝机制.ipynb` ：建树与剪枝机制 + 附录 A（Breiman 排序定理证明）+ 附录 B（1-SE 深探）
 > -  本文件（2） ：按 method 展开 +  附录 C（其余函数与数据集速览） 
 > -  `03_横切专题.ipynb` ：代价敏感、surrogate、重要性、复杂度
 >
 >  前置知识 ：需要先读过  `01_建树与剪枝机制.ipynb`  的 1.2（不纯度与 improve）、1.3–1.4（cp 与 cp 表）、1.5（三棵树对比）。本文件 不再重复  cp 表逐列读法与 1-SE 规则，只引用。

---

## 本文件导读

 `rpart`  根据因变量类型自动选择  `method` ：因子 →  `class` ；连续数值 →  `anova` ；计数 →  `poisson` ； `Surv()`  对象 →  `exp` （生存）。本文件逐个展开：

1.  2.1  `class`  分类树 ：abalone 二分类（平衡）、Gini 与 improve 手算、predict 输出、iris 多分类读图；
2.  2.2  `anova`  回归树 ：$y=x\sin x$ 分段常数拟合、abalone 回归、cp 对 RMSE 的影响；
3.  2.3  `poisson`  计数树 ：solder.balance 焊接缺陷、Poisson 偏差与 improve 手算、必须显式指定 method；
4.  2.4 生存树 ：stagec 前列腺癌、 `Surv(time, status)` 、指数模型偏差下降；
5.  附录 C ：rpart 其余函数与全部数据集速览。

 >  两个约定 ：①  数据勘察 ：每个建模示例前先用  `skimr::skim_tee()`  勘察，并补 skim 不覆盖的派生分析；②  业务背景 ：真实数据集（abalone、solder.balance、stagec、iris）建模前先交代研究问题、变量含义与建模目标。纯模拟数据只说明生成机制。

 >  再次强调 ：计数因变量默认会被当作连续而走  `anova` ，要得到泊松树必须显式  `method = "poisson"` （见 2.3）。 

# 第 2 节 按 method 展开 

 `rpart`  根据因变量类型自动选择  `method` ：因子 →  `class` ；连续数值 →  `anova` ；计数 →  `poisson` ； `Surv()`  对象 →  `exp` （生存）。本节逐个展开，每个 method 只讲它 特有 的部分，cp 表读法、1-SE 规则直接沿用第 1 节。

 >  再次强调 ：计数因变量默认会被当作连续而走  `anova` ，要得到泊松树必须显式  `method = "poisson"` （见 2.3）。

 >  数据勘察约定 ：本节每个建模示例之前都先做一次数据勘察，统一用  `skimr::skim_tee()` ；勘察块与建模块各自自包含。

 >  背景介绍约定 ：本节使用真实业务数据集（abalone、solder.balance、stagec）的地方，建模前先交代数据背景与建模目标。 

## 2.1  `method = "class"` ：分类树 

### 2.1.1 平衡、代价不敏感、混合自变量的二分类 

 问题背景：鲍鱼年龄预测。  鲍鱼是渔业与水产养殖中的重要经济物种，其年龄直接影响肉质、价格与繁殖价值。年龄最准确的测法是把贝壳切片、在显微镜下数环纹（Rings），但这需要杀死鲍鱼、耗时且成本高。现实中更希望 只通过容易测量的物理尺寸与重量，就判断这只鲍鱼属于「老」还是「年轻」 ，以便在不损伤个体的前提下分级定价、估算种群年龄结构。

 数据变量： 
-  因变量  `Rings` ：贝壳环纹数，是年龄的代理（环纹数越多越老）。
-  自变量  `Type` ：鲍鱼性别/类型（公、母、幼），类别型。
-  自变量  `LongestShell` 、 `Diameter` 、 `Height` 、 `WholeWeight` 、 `ShuckedWeight` 、 `VisceraWeight` 、 `ShellWeight` ：尺寸与各部位重量，连续型。

 >  教学点设计：干净、平衡的类别场景 。abalone 原始数据的 Rings 分布是右偏的（年轻的多，老年的少）。为了在第 2 节专心讲解分类树的 基础分裂机制 （Gini、cp、剪枝）， 我们在这里有意将阈值设为  `Rings > 9` 。按此阈值二值化后，L（年轻，≤9）与 H（老， >9）两类样本数量大致维持在  1:1 左右 。在平衡数据下，rpart 的默认行为是完全无偏的，模型不需要处理“倾向多数类”的纠偏问题，它的 全部关注点都在于如何纯粹地切分特征空间 （找哪些尺寸或重量最能区分年龄）。代码中将直接输出频数与比例，用实际数据印证这一“有意设计的平衡”。

 建模目标 ：用尺寸与重量等易测变量，建一棵分类树，把鲍鱼分成「年轻/老」两类；树的分裂规则还能告诉我们 哪些测量指标最能区分年龄 。

abalone 数据集共 4177 只鲍鱼。 先勘察数据，再建模 ；两个代码块各自自包含，独立可运行。 

以下代码 使用  `skimr::skim_tee()`  对 abalone 数据进行全面的初步勘察 ，并在代码中直接输出目标变量二值化后的 类别频数与比例 ，用实际数据印证“有意设计的平衡”。 

In [ ]:
library(rpart)                                       # 加载 rpart 包
library(AppliedPredictiveModeling)                   # 加载 abalone 数据所在包
library(skimr)                                       # 加载 skimr（提供 skim_tee）
library(dplyr)                                       # 加载 dplyr（管道与 mutate）
data(abalone)                                        # 载入 abalone 数据
d_ab   <- abalone %>%                                    # 复制并通过管道处理
  mutate(Rings = factor(Rings > 9,                   # Rings 按 >9 二值化（刻意平衡）
                        labels = c("L", "H")))      # 两类标签 L/H

d_ab %>% skim_tee()                                    # 打印全表摘要并返回原数据（覆盖维度/类型/缺失/分布/水平）
table(d_ab$Rings)                                      # 输出 L/H 两类的绝对样本量
prop.table(table(d_ab$Rings))                          # 输出 L/H 两类的比例（应接近 1:1）
table(d_ab$Type)                                       # 类别自变量 Type 的完整水平分布 

以下代码 执行基础的分类树建模流程 。首先将数据按 50% 划分为训练集和测试集，然后在训练集上使用  `rpart()`  建立一棵充分生长的树（ `cp = 0` ），最后打印出该树的 cp 表以观察其生长情况。 

In [ ]:
library(rpart); library(AppliedPredictiveModeling)   # 加载 rpart 与数据包
data(abalone); d_ab   <- abalone                         # 载入数据并复制
d_ab$Rings   <- factor(d_ab$Rings > 9, labels = c("L", "H"))  # Rings 二值化（平衡）

set.seed(100); i   <- sample(nrow(d_ab), nrow(d_ab) * 0.5) # 随机抽取一半作训练集
tr_ab   <- d_ab[i, ]; te_ab   <- d_ab[-i, ]                  # 划分训练集与测试集

set.seed(2)                                          # 固定种子（xstd 依赖随机切分）
m_ct_ab   <- rpart(Rings ~ ., data = tr_ab, cp = 0)      # 用全部自变量建分类树，充分生长
print(m_ct_ab$cptable)                                 # 打印 cp 表 

cp 表读法同  `01_建树与剪枝机制.ipynb`  1.4。 `summary()`  还会输出各节点的分割详情（Primary splits、Surrogate splits、improve 值）。 

以下代码 使用  `summary()`  函数查看分类树的详细模型摘要 。除了 cp 表外，它还会按节点逐层列出分割详情，包括主分割变量、不纯度下降量（improve）以及用于处理缺失值的代理分割（surrogate splits）。 

In [ ]:
library(rpart); library(AppliedPredictiveModeling)   # 加载 rpart 与数据包
data(abalone); d_ab   <- abalone                         # 载入数据并复制
d_ab$Rings   <- factor(d_ab$Rings > 9, labels = c("L", "H"))  # Rings 二值化（平衡）
set.seed(100); i   <- sample(nrow(d_ab), nrow(d_ab) * 0.5)   # 随机抽一半作训练集
tr_ab   <- d_ab[i, ]; te_ab   <- d_ab[-i, ]                  # 划分训练集与测试集
set.seed(2)                                          # 固定种子
m_ct_ab   <- rpart(Rings ~ ., data = tr_ab, cp = 0)      # 建分类树，充分生长

summary(m_ct_ab)                                       # 查看模型详细摘要 

以下代码 执行完整的 1-SE 剪枝与评估流程 。它先画出交叉验证误差曲线，接着用  `prune_1se()`  提取最优子树，并用  `partykit`  画出最终的树结构图，最后计算并对比该树在训练集和测试集上的误判率。 

In [ ]:
library(rpart); library(partykit)                    # 加载 rpart 与 partykit
library(AppliedPredictiveModeling)                   # 加载数据包
data(abalone); d_ab   <- abalone                         # 载入数据并复制
d_ab$Rings   <- factor(d_ab$Rings > 9, labels = c("L", "H"))  # Rings 二值化（平衡）
set.seed(100); i   <- sample(nrow(d_ab), nrow(d_ab) * 0.5)   # 随机抽一半作训练集
tr_ab   <- d_ab[i, ]; te_ab   <- d_ab[-i, ]                  # 划分训练集与测试集
set.seed(2); m_ct_ab   <- rpart(Rings ~ ., data = tr_ab, cp = 0)  # 建分类树，充分生长

# ---- 1-SE 剪枝函数就地定义 ----
prune_1se   <- function(object, se = 1) {              # 1-SE 剪枝函数
  cptab     <- object$cptable                          # 取出 cp 表
  xerror    <- cptab[, "xerror"]                       # 交叉验证误差均值列
  xstd      <- cptab[, "xstd"]                         # 交叉验证标准差列
  cp        <- cptab[, "CP"]                           # 复杂度参数列
  min_idx   <- which.min(xerror)                       # xerror 最小处下标
  threshold   <- xerror[min_idx] + se * xstd[min_idx]  # 1-SE 阈值
  cp_best   <- max(cp[xerror <= threshold])            # 阈值内取最大 cp
  rpart::prune(object, cp = cp_best)                 # 按 cp 剪枝返回子树
}

plotcp(m_ct_ab); grid()                                # 画出 cp 曲线
m_ct_ab_best   <- prune_1se(m_ct_ab, se = 1)               # 1-SE 剪枝得到最优树
plot(as.party(m_ct_ab_best))                           # 画出最优树结构图
p_cls_tr   <- predict(m_ct_ab_best, tr_ab, type = "class")# 训练集预测类别
p_cls_te   <- predict(m_ct_ab_best, te_ab, type = "class")# 测试集预测类别
c(train_error = mean(p_cls_tr != tr_ab$Rings),        # 训练误差（平衡数据下直接看即可）
  test_error  = mean(p_cls_te != te_ab$Rings))        # 测试误差（平衡数据下直接看即可） 

### 2.1.2  `class`  的 impurity 与 improve 

分类树用  Gini 指数 作不纯度。节点不纯度 $G = 1 - \sum_k p_k^2$，其中 $p_k$ 为节点内第 $k$ 类比例。

一次分割的 improve（分割优度）=  `n·G_parent − (n_L·G_left + n_R·G_right)` 。

仍用 abalone（Rings 二值化）为例：以根节点按  `ShellWeight < 0.2545`  分割，手算验证 improve（ 本块自带数据与 gini 函数 ，独立可运行）： 

以下代码 手工计算并验证 Gini 分割优度（improve） 。代码提取了根节点按  `ShellWeight < 0.2545`  分割时的列联表，并手动实现公式  `n·G_parent − nL·G_left − nR·G_right` ，以验证模型底层的不纯度下降量计算逻辑。 

In [ ]:
library(rpart); library(AppliedPredictiveModeling)   # 加载 rpart 与数据包
data(abalone); d_ab   <- abalone                         # 载入数据并复制
d_ab$Rings   <- factor(d_ab$Rings > 9, labels = c("L", "H"))  # Rings 二值化（平衡）
set.seed(100); i   <- sample(nrow(d_ab), nrow(d_ab) * 0.5)   # 随机抽一半作训练集
tr_ab   <- d_ab[i, ]                                     # 训练集

gini   <- function(n1, n2) 1 - (n1/(n1+n2))^2 - (n2/(n1+n2))^2  # Gini 不纯度函数
tab   <- with(tr_ab, table(cut(ShellWeight,            # 按 ShellWeight 分箱后统计
                             c(0, 0.2545, Inf)),     # 分箱切点 0.2545
                        Rings))                      # 交叉统计类别
print(addmargins(tab))                               # 打印带边际和的列联表
n   <- sum(tab); nL   <- sum(tab[1, ]); nR   <- sum(tab[2, ])  # 总样本、左支、右支样本数
imp   <- n * gini(tab[1,1]+tab[2,1], tab[1,2]+tab[2,2]) -  # 父节点不纯度
       nL * gini(tab[1,1], tab[1,2]) -                 # 左子节点不纯度
       nR * gini(tab[2,1], tab[2,2])                   # 右子节点不纯度
cat("手算 improve = ", round(imp, 4), "\n")           # 输出手算 improve 

与  `summary()`  中根节点  `improve`  一致（数值舍入差异源于软件内部精度）。

 >说明：分类树的分裂准则默认为 Gini 指数；也可通过  `parms = list(split = "information")`  改用信息增益，此时 improve 的计算口径相应变为熵的下降量。 注意 ：在信息增益下，无序离散变量的排序定理不再成立，rpart 会采用更慢的搜索方式。 

### 2.1.3  `class`  的 predict 输出 

-  `type = "class"` ：预测类别（因子）；
-  `type = "prob"` ：各类别概率矩阵（列顺序 =  `levels(y)` ）。

 `r
predict(m_ct_ab_best, te_ab, type = "class") # 因子
predict(m_ct_ab_best, te_ab, type = "prob") # 矩阵，列 = levels
` 

### 2.1.4 多分类：如何读决策树图 

 问题背景：鸢尾花品种分类。  iris 是统计与机器学习中最经典的分类数据集，源自 Fisher（1936）对三种鸢尾花（setosa、versicolor、virginica）的形态测量。现实问题是： 仅凭花瓣与花萼的长宽，能否判断一朵鸢尾属于哪个品种？  这是一个三分类问题，正好用来演示多分类树的读图方式。

 数据变量： 
-  因变量  `Species` ：鸢尾花品种，三类（setosa / versicolor / virginica）。
-  自变量  `Sepal.Length` 、 `Sepal.Width` 、 `Petal.Length` 、 `Petal.Width` ：花萼与花瓣的长度和宽度（cm），连续型。

 >  教学点设计：绝对的、完美的三分类平衡 。iris 数据集天生就是 三类各 50 朵（1:1:1 绝对平衡） 。这是极其难得的“洁净”环境：在多分类中，如果类别不均衡，模型容易把少数类忽略掉。这里三类绝对平衡，使得 rpart 的默认行为完全无偏，模型 唯一的关注点就是如何用特征（主要是花瓣尺寸）在空间中划出纯净的边界 。这使得后续  `partykit`  树图上的各类比例（如 0.9, 0.1, 0.0）能直接反映空间划分的纯度，混淆矩阵的对角线也能直接表明模型在各个品种上的均等识别能力。代码中将直接输出频数验证。

 建模目标 ：用四个形态测量建多分类树，看树如何主要依靠花瓣尺寸把三类分开；叶节点显示各类比例，节点标签为多数类。

iris 共 150 朵花。 先勘察，再建模 ；两个代码块各自自包含，独立可运行。 

以下代码 使用  `skim_tee()`  快速勘察 iris 数据 ，并 直接输出 三类品种的频数与比例，印证绝对平衡。同时查看各类在四个形态测量上的平均值，帮助建立“三类样本在特征空间上是否容易区分”的直觉。 

In [ ]:
library(skimr); library(dplyr)                       # 加载 skimr 与 dplyr
iris %>% skim_tee()                                  # 打印全表摘要并返回原数据
table(iris$Species)                                  # 输出三类绝对样本量（应各为 50）
prop.table(table(iris$Species))                      # 输出三类比例（应各为 1/3）
aggregate(. ~ Species, data = iris[, 1:5], mean)     # 各类在自变量上的均值差异（可分性直觉） 

以下代码 使用  `partykit::plot(as.party())`  画出多分类树图 。通过图上的叶节点方框，可以直观看到三类样本在节点内的比例分布以及最终的预测类别。同时打印了预测概率矩阵和混淆矩阵，方便对照模型效果。 

In [ ]:
library(rpart); library(partykit)                    # 加载 rpart 与 partykit
# ---- 1-SE 剪枝函数就地定义 ----
prune_1se   <- function(object, se = 1) {              # 1-SE 剪枝函数
  cptab     <- object$cptable                          # 取出 cp 表
  xerror    <- cptab[, "xerror"]                       # 交叉验证误差均值列
  xstd      <- cptab[, "xstd"]                         # 交叉验证标准差列
  cp        <- cptab[, "CP"]                           # 复杂度参数列
  min_idx   <- which.min(xerror)                       # xerror 最小处下标
  threshold   <- xerror[min_idx] + se * xstd[min_idx]  # 1-SE 阈值
  cp_best   <- max(cp[xerror <= threshold])            # 阈值内取最大 cp
  rpart::prune(object, cp = cp_best)                 # 按 cp 剪枝返回子树
}
set.seed(2)                                          # 固定种子
m_ct_iris_best   <- prune_1se(rpart(Species ~ ., data = iris, cp = 0), se = 1)  # 建树并 1-SE 剪枝
plot(as.party(m_ct_iris_best))                            # 画多分类树图
head(predict(m_ct_iris_best, iris, type = "prob"), 6)     # 前 6 个样本的三类概率
table(predict(m_ct_iris_best, iris, type = "class"),      # 预测类别与真实类别交叉表
      iris$Species)                                  # （混淆矩阵对照，看对角线识别率） 

 读图要点 ： `as.party()`  画出的每个叶节点方框里会显示三类（setosa / versicolor / virginica）的样本比例与预测类别。例如某个叶节点显示  `(0, 0.9, 0.1)` ，表示该节点内 90% 为 versicolor、10% 为 virginica，节点标签为 versicolor。setosa 与另两类在花瓣尺寸上明显分离，树会优先在  `Petal.Length`  上分割。

 >  多分类与 Gini 的陷阱 ：当响应变量有多类时，Gini 不纯度的排序定理失效。rpart 对无序离散变量会采用其它搜索策略（通常仍是启发式近似），不再保证精确最优。 严格证明与边界讨论见  `01_建树与剪枝机制.ipynb`  的附录 A。 

 >  本节要点 
 > 1.  `class`  用 Gini 不纯度；improve = n·G_parent − Σ n_child·G_child；可用  `parms = list(split="information")`  改用信息增益。
 > 2.  `predict`  的  `type = "class"/"prob"`  分别返回类别/概率矩阵。
 > 3. 多分类叶节点显示各类比例，节点标签为多数类； `type="prob"`  可查看各类概率。
 > 4. 无序离散变量的排序捷径仅二分类 + Gini 或回归 + MSE 有效，多分类与信息增益下失效。 

## 2.2  `method = "anova"` ：回归树 

因变量连续时自动用  `anova` 。不纯度用 方差（SSE） ，improve =  `SST − (SSL + SSR)` 。 

### 2.2.1 直观：回归曲线 

 生成机制 ：真实曲线 $y = x\sin(x)$ 加噪声，树用分段常数逼近。这是纯模拟数据，用来直观展示回归树如何用阶梯状分段拟合非线性曲线。 先勘察生成的训练/测试数据，再建模 ；两个代码块各自自包含，独立可运行。 

以下代码 按理论曲线生成回归任务的训练集和测试集 ，并使用  `skim_tee()`  对两套数据进行初步的统计摘要，同时核对它们在 x 轴上的取值范围是否一致，以确保后续评估的公平性。 

In [ ]:
library(skimr); library(dplyr)                       # 加载 skimr 与 dplyr
n   <- 1000                                            # 训练/测试集各自的样本量
set.seed(100)                                        # 训练集种子
x   <- sort(runif(n, -2 * pi, pi))                       # 训练集 x，排序
y   <- x * sin(x) + 1.2 * rnorm(n)                         # 训练集 y，按理论曲线加噪声
sim_re_tr   <- data.frame(x, y)                            # 训练集
set.seed(200)                                        # 测试集种子
x   <- runif(n, -2 * pi, pi)                             # 测试集 x
y   <- x * sin(x) + 1.2 * rnorm(n)                         # 测试集 y
sim_re_te   <- data.frame(x, y)                            # 测试集

sim_re_tr %>% skim_tee()                                 # 训练集全表摘要并返回原数据
sim_re_te %>% skim_tee()                                 # 测试集全表摘要并返回原数据
range(sim_re_tr$x); range(sim_re_te$x)                       # 训练/测试 x 覆盖范围是否一致（skim 给 min/max，这里并排对照） 

以下代码 构建回归树并将其预测结果可视化 。代码建树并剪枝后，把树在训练集上的分段常数预测值画成蓝色折线，叠加在散点图上，与真实的红色理论曲线对比，直观展示回归树用阶梯状分段拟合平滑曲线的特点。 

In [ ]:
library(rpart)                                       # 加载 rpart 包
n   <- 1000                                            # 训练/测试集各自的样本量
set.seed(100)                                        # 训练集种子
x   <- sort(runif(n, -2 * pi, pi))                       # 训练集 x，排序
y   <- x * sin(x) + 1.2 * rnorm(n)                         # 训练集 y，按理论曲线加噪声
sim_re_tr   <- data.frame(x, y)                            # 训练集
set.seed(200)                                        # 测试集种子
x   <- runif(n, -2 * pi, pi)                             # 测试集 x
y   <- x * sin(x) + 1.2 * rnorm(n)                         # 测试集 y
sim_re_te   <- data.frame(x, y)                            # 测试集

# ---- 1-SE 剪枝函数就地定义 ----
prune_1se   <- function(object, se = 1) {              # 1-SE 剪枝函数
  cptab     <- object$cptable                          # 取出 cp 表
  xerror    <- cptab[, "xerror"]                       # 交叉验证误差均值列
  xstd      <- cptab[, "xstd"]                         # 交叉验证标准差列
  cp        <- cptab[, "CP"]                           # 复杂度参数列
  min_idx   <- which.min(xerror)                       # xerror 最小处下标
  threshold   <- xerror[min_idx] + se * xstd[min_idx]  # 1-SE 阈值
  cp_best   <- max(cp[xerror <= threshold])            # 阈值内取最大 cp
  rpart::prune(object, cp = cp_best)                 # 按 cp 剪枝返回子树
}

set.seed(2); m_rt   <- rpart(y ~ ., data = sim_re_tr, cp = 0) # 建回归树，充分生长
plotcp(m_rt); grid()                                   # 画 cp 曲线
m_rt_best   <- prune_1se(m_rt, se = 1)                     # 1-SE 剪枝

plot(sim_re_tr$x, sim_re_tr$y, cex = 0.5, xlab = "x", ylab = "y")  # 画训练集散点
lines(sim_re_tr$x, predict(m_rt_best, sim_re_tr),              # 叠加 1-SE 树预测曲线
      lwd = 4, col = "blue")                         # 线宽与颜色
curve(x * sin(x), -2 * pi, pi, lwd = 6, col = "red", lty = 3, add = TRUE)  # 叠加理论曲线
legend("topright", c("理论", "1-SE 回归树"),        # 图例
       lwd = c(4, 6), col = c("red", "blue"), lty = c(3, 1))
grid()                                               # 网格线 

### 2.2.2  `anova`  的 predict 输出 

-  `type = "vector"` （默认）：预测均值向量；
-  `type = "matrix"` ： `nsplit × n`  矩阵。

 `r
predict(m_rt_best, sim_re_te) # 向量
predict(m_rt_best, sim_re_te, type="matrix") # 矩阵
` 

### 2.2.3 自变量混合取值：abalone 回归 

 问题背景（回归口径） ：仍是鲍鱼年龄问题，但这次不二值化——直接 用尺寸与重量回归预测环纹数（年龄的连续代理） 。与 2.1.1 的二分类相比，回归口径保留更多信息：不只判断「老/年轻」，而是估计「大概多少环」。这对估算种群年龄结构、分级定价更有用。

 数据变量 ：因变量  `Rings`  保持数值（1–29）；自变量同 2.1.1（ `Type`  类别型 + 七个尺寸/重量连续型）。

 建模目标 ：建回归树预测环纹数，用 RMSE 衡量误差；树的规则告诉我们哪些测量指标对年龄贡献最大。

 先勘察回归口径的数据，再建模 ；两个代码块各自自包含，独立可运行。 

以下代码 针对连续因变量（ `Rings` ）进行建模前的特征勘察 。除了使用  `skim_tee()`  打印全表摘要外，还专门调用了  `summary()`  和  `var()`  查看因变量的分布与方差，绘制直方图观察偏态，并检查  `Type`  水平分布。 

In [ ]:
library(AppliedPredictiveModeling); library(skimr)  # 加载数据包与 skimr
library(dplyr)                                       # 加载 dplyr
data(abalone); d_ab2   <- abalone                        # 载入数据并复制（Rings 保持数值）

d_ab2 %>% skim_tee()                                   # 打印全表摘要并返回原数据
summary(d_ab2$Rings); var(d_ab2$Rings)                   # 因变量分布与方差（skim 不给方差）
hist(d_ab2$Rings, breaks = 30)                         # 因变量直方图（看偏态，skim 不给图形）
table(d_ab2$Type)                                      # 类别自变量完整水平分布 

以下代码 执行回归树的完整建模、剪枝与评估流程 。建树和 1-SE 剪枝后，绘制树结构图，最后计算该树在训练集和测试集上的 RMSE（均方根误差），以此来评估回归预测的准确度。 

In [ ]:
library(rpart); library(partykit); library(AppliedPredictiveModeling)  # 加载 rpart、partykit、数据包
data(abalone); d_ab2   <- abalone                        # 载入数据并复制（Rings 保持数值）
set.seed(100); I   <- sample(nrow(d_ab2), nrow(d_ab2)*0.5) # 随机抽一半作训练集
tr_ab2   <- d_ab2[I, ]; te_ab2   <- d_ab2[-I, ]                    # 划分训练集与测试集

# ---- 1-SE 剪枝函数就地定义 ----
prune_1se   <- function(object, se = 1) {              # 1-SE 剪枝函数
  cptab     <- object$cptable                          # 取出 cp 表
  xerror    <- cptab[, "xerror"]                       # 交叉验证误差均值列
  xstd      <- cptab[, "xstd"]                         # 交叉验证标准差列
  cp        <- cptab[, "CP"]                           # 复杂度参数列
  min_idx   <- which.min(xerror)                       # xerror 最小处下标
  threshold   <- xerror[min_idx] + se * xstd[min_idx]  # 1-SE 阈值
  cp_best   <- max(cp[xerror <= threshold])            # 阈值内取最大 cp
  rpart::prune(object, cp = cp_best)                 # 按 cp 剪枝返回子树
}

set.seed(2); m_rt_ab   <- rpart(Rings ~ ., data = tr_ab2, cp = 0)  # 建回归树，充分生长
plotcp(m_rt_ab); grid()                                # 画 cp 曲线
m_rt_ab_best   <- prune_1se(m_rt_ab, se = 1)               # 1-SE 剪枝
plot(as.party(m_rt_ab_best))                           # 画剪枝后的树

rmse   <- function(pred, obs) sqrt(mean((pred - obs)^2))  # 定义 RMSE
c(train_RMSE = rmse(predict(m_rt_ab_best, tr_ab2), tr_ab2$Rings),  # 训练集 RMSE
  test_RMSE  = rmse(predict(m_rt_ab_best, te_ab2), te_ab2$Rings))  # 测试集 RMSE 

### 2.2.4 cp 对回归性能的影响 

遍历 cp 表各 cp 值，画训练/测试 RMSE 曲线，验证 1-SE 最优 cp 的合理性。 本块自带数据、建模与 rmse 函数 ，独立可运行（数据勘察见 2.2.3，口径相同，此处不重复）。 

以下代码 使用  `plotly`  绘制动态交互图 ，直观展示 cp 值对回归树性能的影响。代码遍历不同的 cp 值重建模型并计算 RMSE，将训练集（黑线）和测试集（红线）的误差曲线画在同一张图上，验证 1-SE 规则选出的 cp 恰好位于测试误差的拐点附近。 

In [ ]:
library(rpart); library(plotly); library(AppliedPredictiveModeling)  # 加载 rpart、plotly、数据包
data(abalone); d_ab2   <- abalone                        # 载入数据并复制
set.seed(100); I   <- sample(nrow(d_ab2), nrow(d_ab2)*0.5) # 随机抽一半作训练集
tr_ab2   <- d_ab2[I, ]; te_ab2   <- d_ab2[-I, ]                    # 划分训练集与测试集
rmse   <- function(pred, obs) sqrt(mean((pred - obs)^2))  # 定义 RMSE
set.seed(2); m_rt_ab   <- rpart(Rings ~ ., data = tr_ab2, cp = 0)  # 建回归树，充分生长

cp   <- m_rt_ab$cptable; a   <- cp[, 1]; L   <- length(a)   # 取 cp 列与长度
Cp   <- sqrt(a[-L] * a[-1])                            # 相邻 cp 的几何平均
err_tr   <- err_te   <- numeric(length(Cp))              # 预分配训练/测试误差向量
for (k in seq_along(Cp)) {                           # 遍历每个 cp 值
  m   <- rpart(Rings ~ ., data = tr_ab2, cp = Cp[k])      # 以该 cp 重新建树
  err_tr[k]   <- rmse(predict(m, tr_ab2), tr_ab2$Rings)      # 训练集 RMSE
  err_te[k]   <- rmse(predict(m, te_ab2), te_ab2$Rings)      # 测试集 RMSE
}
plot_ly(x = Cp) %>%                                  # 以 cp 为横轴画交互图
  add_trace(y = err_tr, type = "scatter",            # 训练集曲线
            name = "train", mode = "lines+markers", color = I("black")) %>%
  add_trace(y = err_te, type = "scatter",            # 测试集曲线
            name = "test",  mode = "lines+markers", color = I("red")) %>%
  layout(title = "RMSE vs cp",                        # 标题
         xaxis = list(title = "cp"),                  # 横轴标签
         yaxis = list(title = "RMSE"))                # 纵轴标签 

 >  本节要点 
 > 1.  `anova`  用 SSE/方差作不纯度；improve = SST − (SSL+SSR)。
 > 2.  `predict`  默认  `type = "vector"`  返回均值； `"matrix"`  返回节点级矩阵。
 > 3. cp 越小训练 RMSE 越低、测试 RMSE 先降后升；1-SE cp 位于测试曲线拐点附近。
 > 4. 无序离散变量的排序定理在回归 + MSE 下成立，rpart 依然可以高效处理。 

## 2.3  `method = "poisson"` ：计数因变量 

因变量为计数时用  `poisson` 。improve = 似然比统计量： `D_parent − (D_left + D_right)` ，D 为 Poisson 偏差。

 >  必须显式指定 ：计数因变量若直接写数值， `rpart()`  会自动走  `anova` ，不会自动选  `poisson` 。

 问题背景：印刷电路板焊接缺陷。  在电子制造中，印刷电路板（PCB）的焊接质量直接决定产品良率。焊接工艺由多个因素共同决定：焊盘开口大小（Opening）、焊料类型（Solder）、掩膜类型（Mask）、焊盘类型（PadType）、所在面板（Panel）。工程上关心的是： 在给定的工艺条件下，一块板子上平均会出现多少个焊接缺陷（skips）？  缺陷数是非负整数计数，正是泊松树该处理的场景。

 数据变量： 
-  因变量  `skips` ：每块板子的焊接缺陷数（0, 1, 2, …），计数型。
-  自变量  `Opening` 、 `Solder` 、 `Mask` 、 `PadType` 、 `Panel` ：五个工艺条件，全部类别型。

 建模目标 ：建泊松树估计各工艺条件下的平均缺陷数 λ；树的规则告诉我们 哪些工艺条件组合会显著抬高缺陷率 ，从而指导工艺改进。

solder.balance 是 solder 数据的平衡版本（各条件组合样本量更均衡）。 先勘察数据，再建模 ；以下各代码块各自自包含，独立可运行。 

以下代码 针对计数型因变量进行专门的建模前勘察 。除了基本的摘要输出，重点检查了  `skips`  的零膨胀情况、均值与方差的关系（泊松假设），并按不同的工艺条件分组计算了缺陷均值，初步排查哪些条件对缺陷数影响较大。 

In [ ]:
library(rpart)                                       # 加载 rpart 包（solder.balance 随包提供）
library(skimr); library(dplyr)                       # 加载 skimr 与 dplyr

solder.balance %>% skim_tee()                        # 打印全表摘要并返回原数据（覆盖维度/类型/缺失/分布/水平）
summary(solder.balance$skips)                        # 因变量范围与分位数
table(solder.balance$skips)                          # 各计数值的频数（看零膨胀）
mean(solder.balance$skips); var(solder.balance$skips)  # 均值与方差（泊松假设下应接近相等；skim 不给方差）

# 各条件下 skips 的均值（skim 不做分组，必须保留）
aggregate(skips ~ Opening, data = solder.balance, FUN = mean)  # 按 Opening 分组均值
aggregate(skips ~ Solder,  data = solder.balance, FUN = mean)  # 按 Solder 分组均值
aggregate(skips ~ Mask,    data = solder.balance, FUN = mean)  # 按 Mask 分组均值 

 判读要点 ：泊松假设下均值与方差应接近；若方差明显大于均值（过离散），严格说应改用负二项等模型。 `pois_dev`  里对  `y == 0`  的特殊处理，正因为零膨胀常见（很多板子一个缺陷都没有）。 `aggregate`  的均值差异告诉你哪些工艺条件可能成为树的首选分割变量。 

以下代码 建泊松树并进行剪枝和可视化 。注意这里必须显式指定  `method = "poisson"` ，否则计数型因变量会被错误地当成连续值处理。剪枝后画出树图，并查看  `summary()`  以获得根节点的 improve 值。 

In [ ]:
library(rpart); library(partykit)                    # 加载 rpart 与 partykit
# ---- 1-SE 剪枝函数就地定义 ----
prune_1se   <- function(object, se = 1) {              # 1-SE 剪枝函数
  cptab     <- object$cptable                          # 取出 cp 表
  xerror    <- cptab[, "xerror"]                       # 交叉验证误差均值列
  xstd      <- cptab[, "xstd"]                         # 交叉验证标准差列
  cp        <- cptab[, "CP"]                           # 复杂度参数列
  min_idx   <- which.min(xerror)                       # xerror 最小处下标
  threshold   <- xerror[min_idx] + se * xstd[min_idx]  # 1-SE 阈值
  cp_best   <- max(cp[xerror <= threshold])            # 阈值内取最大 cp
  rpart::prune(object, cp = cp_best)                 # 按 cp 剪枝返回子树
}
set.seed(2)                                          # 固定种子
m_pois   <- rpart(skips ~ Opening + Solder + Mask + PadType + Panel,  # 建泊松树
              data = solder.balance,                 # 数据
              method = "poisson")                    # 显式指定泊松
plotcp(m_pois); grid()                                 # 画 cp 曲线
m_pois_best   <- prune_1se(m_pois, se = 1)                     # 1-SE 剪枝
plot(as.party(m_pois_best))                                # 画剪枝后的树
summary(m_pois)                                        # 查看根节点 improve 

 improve 的手算对照 ：泊松树用 Poisson 偏差 D 衡量不纯度。对节点内计数 $y_i$、均值 $\bar y$，偏差 $D = 2\sum [y_i\log(y_i/\bar y) - (y_i-\bar y)]$（$y_i=0$ 项取 0）。一次分割的 improve =  `D_parent − (D_left + D_right)` 。 本块自带数据与 pois_dev 函数 ，独立可运行： 

以下代码 手工计算并验证泊松树中根节点的 Poisson 偏差下降量（improve） 。代码定义了 Poisson 偏差公式，提取根节点的切分方式拆分数据，分别计算父节点和左右子节点的偏差，做减法后与模型输出的 improve 值进行核对。 

In [ ]:
library(rpart)                                       # 加载 rpart 包
set.seed(2)                                          # 固定种子
m_pois   <- rpart(skips ~ Opening + Solder + Mask + PadType + Panel,  # 建泊松树
              data = solder.balance,                 # 数据
              method = "poisson")                    # 显式指定泊松
pois_dev   <- function(y) {                            # Poisson 偏差函数
  ybar   <- mean(y)                                    # 节点内计数均值
  if (ybar == 0) return(0)                           # 均值为 0 时偏差为 0
  2 * sum(ifelse(y == 0, 0, y * log(y / ybar)) - (y - ybar))  # 偏差公式（y=0 项取 0）
}
first_var   <- "Opening"                               # 取根节点第一分割变量
lv   <- levels(solder.balance[[first_var]])            # 该变量的水平
idx   <- summary(m_pois)$splits[1, "index"]               # 左支水平编码
left_lv   <- lv[idx]                                   # 左支水平名
left   <- solder.balance[[first_var]] %in% left_lv     # 左支样本指示
y   <- solder.balance$skips                            # 计数向量
imp   <- pois_dev(y) - pois_dev(y[left]) - pois_dev(y[!left])  # 父偏差减左右子偏差
cat("手算 improve = ", round(imp, 4), "\n")           # 输出手算 improve 

 >提示：solder.balance 的自变量均为类别型，rpart 对类别变量会在水平组合中搜索最优二分； `summary()$splits`  的  `index`  编码了左支水平，具体拆分方式以  `summary()`  输出的方向说明为准。手算结果与  `summary()`  的 improve 在同一量级即可验证口径正确。

 `poisson`  的  `predict`  输出（ 本块自带数据，独立可运行 ）： 

以下代码 查看泊松树模型的两种预测输出格式 。默认的  `type = "vector"`  会返回每个样本预测的缺陷数均值 λ，而  `type = "matrix"`  则返回一个节点级的预测矩阵，包含每个样本落入不同叶节点后的预测均值。 

In [ ]:
library(rpart)                                       # 加载 rpart 包
# ---- 1-SE 剪枝函数就地定义 ----
prune_1se   <- function(object, se = 1) {              # 1-SE 剪枝函数
  cptab     <- object$cptable                          # 取出 cp 表
  xerror    <- cptab[, "xerror"]                       # 交叉验证误差均值列
  xstd      <- cptab[, "xstd"]                         # 交叉验证标准差列
  cp        <- cptab[, "CP"]                           # 复杂度参数列
  min_idx   <- which.min(xerror)                       # xerror 最小处下标
  threshold   <- xerror[min_idx] + se * xstd[min_idx]  # 1-SE 阈值
  cp_best   <- max(cp[xerror <= threshold])            # 阈值内取最大 cp
  rpart::prune(object, cp = cp_best)                 # 按 cp 剪枝返回子树
}
set.seed(2)                                          # 固定种子
m_pois   <- rpart(skips ~ Opening + Solder + Mask + PadType + Panel,  # 建泊松树
              data = solder.balance,                 # 数据
              method = "poisson")                    # 显式指定泊松
m_pois_best   <- prune_1se(m_pois, se = 1)                     # 1-SE 剪枝
p_pois   <- predict(m_pois_best, solder.balance)               # 默认 vector：预测计数均值 λ
head(p_pois)                                           # 查看前几个预测值
pm   <- predict(m_pois_best, solder.balance, type = "matrix")  # matrix：节点级矩阵
dim(pm)                                              # 矩阵维度 

 >  本节要点 
 > 1. 建模前先勘察计数因变量：看零膨胀、均值-方差关系、类别自变量水平。
 > 2.  `poisson`  用 Poisson 偏差的似然比统计量作 improve： `D_parent − (D_left + D_right)` ；计数因变量必须显式指定  `method = "poisson"` 。
 > 3.  `predict`  默认  `type = "vector"`  返回预测计数均值（实数 λ）； `"matrix"`  返回节点级矩阵。
 > 4. cp 表读法、1-SE 规则与 class/anova 完全一致。 

## 2.4 生存分析： `Surv()`  因变量 

因变量为  `survival::Surv(time, status)`  时，rpart 自动用  `method = "exp"` （指数生存模型）。 `Surv`  参数： `time`  随访时间， `event`  状态（0=存活/删失，1=死亡）。

生存树的 improve 基于 指数模型的偏差（deviance）下降 ：一次分割的 improve =  `D_parent − (D_left + D_right)` 。

 问题背景：前列腺癌患者生存分析。  stagec 数据来自一项前列腺癌 Stage C 患者的临床随访研究。临床关心的是： 患者确诊后能存活多久，哪些因素（年龄、肿瘤分级、倍体类型等）会显著影响生存？  这类数据的难点在于 删失 ——随访结束时部分患者仍存活，我们只知道他们「至少活到了随访终点」，不知道确切死亡时间。生存树正是为这种「时间至事件 + 删失」结构设计的。

 数据变量： 
-  因变量  `Surv(pgtime, pgstat)` ： `pgtime`  为随访时间（月）， `pgstat`  为事件状态（1=死亡，0=删失）。
-  自变量  `age` 、 `eet` 、 `g2` 、 `grade` 、 `gleason` 、 `ploidy` ：年龄、治疗史、肿瘤分级、Gleason 评分、倍体类型等临床与病理指标（连续与类别混合）。

 建模目标 ：建生存树，把患者按预后风险分组；每个终端节点输出一条生存曲线，可用来识别高风险人群、指导治疗强度。

 先勘察生存数据，再建模 ；以下代码块各自自包含，独立可运行。 

以下代码 对含有生存时间与状态的数据进行专门勘察 。重点查看了删失比例、随访时间的整体分布，并对比了事件组和删失组的随访时间差异，以评估数据的删失程度对后续生存分析建模的潜在影响。 

In [ ]:
library(rpart); library(survival)                    # 加载 rpart 与 survival（stagec 随 rpart 提供）
library(skimr); library(dplyr)                       # 加载 skimr 与 dplyr

stagec %>% skim_tee()                                # 打印全表摘要并返回原数据
table(stagec$pgstat)                                 # 事件状态分布（1=死亡，0=删失；skim 只给 top_counts）
mean(stagec$pgstat == 0)                             # 删失比例（skim 不直接给）
summary(stagec$pgtime)                               # 随访时间范围
hist(stagec$pgtime, breaks = 30)                     # 随访时间直方图（skim 不给图形）

# 事件组 vs 删失组的随访时间对比（skim 不做分组，必须保留）
aggregate(pgtime ~ pgstat, data = stagec, FUN = summary)  # 按状态分组摘要随访时间 

 判读要点 ：删失比例不低时，终端节点的 Kaplan-Meier 曲线能否稳定估计，取决于该节点内事件数与随访时间跨度； `pgtime`  范围决定生存曲线横轴。若删失集中在较早阶段，晚期生存估计会不稳定。 

以下代码 完成生存树的建模、剪枝与预测展示 。使用  `Surv()`  构建生存因变量后， `rpart()`  会自动采用  `method = "exp"` 。代码展示了默认的预测生存时间向量，以及节点级的预测矩阵输出格式。 

In [ ]:
library(rpart); library(partykit); library(survival) # 加载 rpart、partykit、survival
# ---- 1-SE 剪枝函数就地定义 ----
prune_1se   <- function(object, se = 1) {              # 1-SE 剪枝函数
  cptab     <- object$cptable                          # 取出 cp 表
  xerror    <- cptab[, "xerror"]                       # 交叉验证误差均值列
  xstd      <- cptab[, "xstd"]                         # 交叉验证标准差列
  cp        <- cptab[, "CP"]                           # 复杂度参数列
  min_idx   <- which.min(xerror)                       # xerror 最小处下标
  threshold   <- xerror[min_idx] + se * xstd[min_idx]  # 1-SE 阈值
  cp_best   <- max(cp[xerror <= threshold])            # 阈值内取最大 cp
  rpart::prune(object, cp = cp_best)                 # 按 cp 剪枝返回子树
}
set.seed(2)                                          # 固定种子
m_surv   <- rpart(survival::Surv(pgtime, pgstat) ~       # 生存因变量
                age + eet + g2 + grade + gleason + ploidy,  # 自变量
              data = stagec)                         # 数据
m_surv$method                                          # 确认 method 为 exp
plotcp(m_surv); grid()                                 # 画 cp 曲线
m_surv_best   <- prune_1se(m_surv, se = 1)                     # 1-SE 剪枝
plot(as.party(m_surv_best))                                # 画剪枝后的树
p_surv   <- predict(m_surv_best, stagec)                         # 默认 vector：预测生存时间
head(p_surv)                                             # 查看前几个预测值
sm   <- predict(m_surv_best, stagec, type = "matrix"); dim(sm)  # 节点级矩阵及其维度 

 >  本节要点 
 > 1. 生存数据建模前必看删失比例、随访时间分布与事件/删失两组的时间对比。
 > 2. 因变量写  `Surv(time, event)` ，method 自动为  `exp` ；improve 基于指数模型偏差下降。
 > 3.  `predict`  默认  `type = "vector"`  返回预测生存时间； `"matrix"`  返回节点级矩阵。
 > 4. 其余流程（cp 表、1-SE、 `prune_1se` ）不变。 

---

 <a id="appendix-c"> </a>
# 附录 C. rpart 其余函数与数据集速览（补全包内全部对象）

正文只讲了主干函数。本节把  `rpart`  包内 未被正文展开 的函数与数据集按用途归类速览，保证对包内全部对象有交代；每一项只给一句话用途，需要时再查  `?函数名` 。

### C.1 诊断类（看模型哪里不对）

| 函数 | 用途 | 典型用法 |
|---|---|---|
|  `residuals.rpart()`  | 提取每个样本的残差（分类为误判指示，回归为观测减预测，泊松/生存为偏差残差） |  `resid(fit)`  |
|  `meanvar.rpart()`  /  `meanvar()`  | 画各节点的均值-方差图，观察不同节点的响应分布是否同质 |  `meanvar(fit)`  |
|  `rsq.rpart()`  | 画不同分裂数下的近似 R²，直观看到「再加分裂还能提升多少」 |  `rsq.rpart(fit)`  |

### C.2 交叉验证类（与 cp 表同源）

| 函数 | 用途 | 典型用法 |
|---|---|---|
|  `xpred.rpart()`  | 返回交叉验证预测值（cp 表里的 xerror 就是由它汇总而来），可用于自定义 CV 诊断 |  `xpred.rpart(fit, cp = 0.01)`  |

### C.3 绘图底层类（plot/text 的零件）

| 函数 | 用途 | 典型用法 |
|---|---|---|
|  `labels.rpart()`  | 生成每个节点的分割标签文本， `text.rpart`  内部即调用它 |  `labels(fit)`  |
|  `post.rpart()`  /  `post()`  | 输出 PostScript 格式树图（老式打印用，现代环境建议用  `rpart.plot`  /  `partykit`  替代） |  `post(fit, file = "tree.ps")`  |

### C.4 内部/辅助类（了解即可）

| 函数/对象 | 用途 |
|---|---|
|  `na.rpart()`  | 建树前处理缺失值的内部函数：标记哪些行因缺失被排除，是  `03_横切专题.ipynb`  3.2 节缺失机制的底层实现 |
|  `rpart.exp()`  | 指数（生存）拟合的初始化函数，2.4 节  `method = "exp"`  的底层入口 |
|  `rpart.object`  | 树对象的结构定义（ `$cptable` 、 `$variable.importance` 、 `$splits`  等字段的文档），读对象字段时查它 |

### C.5 数据集补全

正文用过  `solder.balance` （2.3）、 `stagec` （2.4）， `00_总览与环境.ipynb`  0.2 已列  `kyphosis` 、 `solder` 、 `car.test.frame` 。包内 全部数据集 如下：

| 数据集 | 内容 | 教程中的使用 |
|---|---|---|
|  `kyphosis`  | 儿童脊柱矫正手术后是否出现后凸畸形的数据（二分类经典例） | 仅列名 |
|  `solder`  /  `solder.balance`  | 印刷电路板焊接缺陷数据； `solder.balance`  为其平衡版本 | 2.3 泊松树 |
|  `stagec`  | 前列腺癌 Stage C 生存数据（含随访时间与状态） | 2.4 生存树 |
|  `car.test.frame`  | 《Consumer Reports》1990 汽车数据（测试版） | 仅列名 |
|  `car90`  | 同源汽车数据（更完整版本，含更多变量与缺失） | 未用 |
|  `cu.summary`  | 同源汽车数据的摘要子集 | 未用 |

 >说明： `car.test.frame` 、 `car90` 、 `cu.summary`  是同一组汽车数据的三个版本，选一个用于练习即可； `kyphosis`  是二分类树的最小示例，适合快速试手。

### C.6 小结

-  主干已讲透 ： `rpart` 、 `rpart.control` 、 `predict` 、 `printcp`  /  `plotcp` 、 `prune` 、 `summary` 、 `print` 、 `plot`  /  `text` 、 `path`  /  `snip` 。
-  本节补全 ： `residuals` 、 `meanvar` 、 `rsq.rpart` 、 `xpred` 、 `labels` 、 `post` 、 `na.rpart` 、 `rpart.exp` 、 `rpart.object` ，以及  `car90` 、 `cu.summary`  数据集。
-至此， `rpart`  包内 全部函数与数据集 均有交代：常用的在正文深入，少用的在此速览，按需回查  `?函数名`  即可。 

---

## 本文件小结与下一步

本文件按 method 逐个展开： `class` （Gini 与 improve 手算、predict 输出、iris 多分类读图）、 `anova` （分段常数拟合、abalone 回归、cp 对 RMSE 的影响）、 `poisson` （Poisson 偏差与 improve 手算、必须显式指定 method）、生存树（ `Surv`  因变量、删失勘察、指数偏差下降），并以 附录 C  补全了 rpart 包内其余函数与全部数据集。

 下一步 ：进入  `03_横切专题.ipynb` 。该文件讲与 method 正交的四个专题：代价敏感三法（先验/代价/权重）、缺失值与 surrogate、变量重要性（自带 vs permutation）、时间复杂度。 